In [ ]:
import os
import requests
from io import BytesIO
from PIL import Image
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import requests
from io import BytesIO

# ==============================================================================
# 2. DATA PREPARATION
# This part downloads your image, splits it, labels it, and saves it.
# ==============================================================================

# Define labels for our context vector
# ==============================================================================
# 1. DEFINE YOUR LABELS (The "Schema")
# This list defines the columns of our final label matrix.
# ==============================================================================
# --- CORRECTED LABEL_NAMES ---
LABEL_NAMES = [
    # 1. Color (Usually one per emoji)
    'color_yellow', 'color_red', 'color_green', 'color_blue', 'color_purple',
    'color_pink', 'color_white', 'color_orange', 'color_brown', 'color_black',
    'color_rainbow',

    # 2. Emotion
    'emotion_happy', 'emotion_laughing', 'emotion_sad', 'emotion_cry',
    'emotion_angry', 'emotion_neutral', 'emotion_surprise', 'emotion_love',
    'emotion_sleepy', 'emotion_shy', 'emotion_evil', 'emotion_scared', 'emotion_sick',
    'emotion_goofy', 'emotion_smirk', 'emotion_yawn', 'emotion_dizzy',

    # 3. Objects / Accessories (Can have multiple)
    'object_sunglasses', 'object_glasses', 'object_heart', 'object_tears',
    'object_halo', 'object_horns', 'object_domino_mask', 'object_sick_mask',
    'object_thermometer', 'object_money', 'object_hat_cowboy', 'object_hat_party',
    'object_grawlix', 'object_zz', 'object_bandage', 'object_monocle',
    'object_hand',

    # 4. Characters / Entities (Usually one per emoji)
    'entity_cyclops', 'entity_clown', 'entity_devil', 'entity_ghost',
    'entity_alien', 'entity_skull', 'entity_poop', 'entity_pumpkin',
    'entity_no_face', 'entity_warning', 'entity_crossed_out', 'entity_tick',
    'entity_robot',

    # 5. Style / Expression Details
    'mouth_kiss', 'mouth_tongue', 'mouth_zigzag', 'mouth_bend', 'mouth_teeth',
    'mouth_zip', 'mouth_drool',
    'eyebrow_raise', 'eyes_xx', 'eyes_evil', 'eyes_wink', 'eyes_closed',
    'eyes_money', 'eyes_roll', 'eyes_empty', 'eyes_shine', 'eyes_star',
    'eyes_wide', 'eyes_spiral',
    'upside_down', 'long_nose', 'wide_nose',

    # 6. Effects (Visual flourishes)
    'effect_vomit', 'effect_sweat', 'effect_tears', 'effect_steam', 'effect_freeze',
    'effect_exploding_head', 'effect_runny_nose', 'style_outline'
]

N_CLASSES = len(LABEL_NAMES)


# ==============================================================================
# 2. PREPARE THE DATASET (The Best Way)
# ==============================================================================
def prepare_dataset(sprite_size=16):
    """
    Fetches emoji files, applies labels, and performs data augmentation
    by duplicating rare-colored emojis to balance the dataset.
    """
    # Caching check remains the same
    if os.path.exists('emojis.npy') and os.path.exists('labels.npy'):
        print("Dataset files already exist. Loading from disk.")
        sprites = np.load('emojis.npy')
        labels = np.load('labels.npy')
        return sprites, labels

    # --- GitHub fetching remains the same ---
    # (Your existing GitHub fetching code goes here... I've omitted it for brevity)
    api_url = "https://api.github.com/repos/cbarkinozer/DataScience/contents/LargeLanguageModels/Emoji1"
    print(f"Fetching dataset info from GitHub API: {api_url}")
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        repo_contents = response.json()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Failed to fetch data from GitHub API: {e}")
    image_urls = sorted([item['download_url'] for item in repo_contents if item['name'].lower().endswith('.png')])
    if not image_urls:
        raise FileNotFoundError("No .png files found in the specified GitHub repository folder.")
    print(f"Found {len(image_urls)} emoji images. Downloading...")
    sprites = []
    for url in tqdm(image_urls, desc="Downloading images"):
        img_response = requests.get(url)
        sprite = Image.open(BytesIO(img_response.content)).convert("RGB")
        if sprite.size != (sprite_size, sprite_size):
            sprite = sprite.resize((sprite_size, sprite_size), Image.Resampling.LANCZOS)
        sprites.append(np.array(sprite))
    sprites = np.array(sprites)


    # ==============================================================================
    # THE MASTER LABELING DICTIONARY
    # THIS IS THE ONLY SECTION YOU NEED TO EDIT.
    # Go one by one and define the labels for each image index.
    # ==============================================================================
    # ==============================================================================
    # THE MASTER LABELING DICTIONARY (FULLY CORRECTED AND EXPANDED)
    # ==============================================================================
    image_label_map = {
        # --- Standard Yellow Emojis ---
        0:  ['color_yellow', 'emotion_happy'],
        1:  ['color_yellow', 'emotion_neutral'],
        2:  ['color_yellow', 'emotion_sad'],
        3:  ['color_yellow', 'emotion_happy'],
        4:  ['color_yellow', 'emotion_happy'],
        5:  ['color_yellow', 'emotion_happy'],
        6:  ['color_yellow', 'emotion_happy'],
        7:  ['color_yellow', 'emotion_neutral'],
        8:  ['color_yellow', 'emotion_sad'],
        9:  ['color_yellow', 'emotion_sad'],
        10: ['color_yellow', 'emotion_surprise'],
        11: ['color_yellow', 'emotion_neutral', 'upside_down'],
        12: ['color_yellow', 'emotion_happy', 'upside_down'],
        13: ['color_yellow', 'emotion_sad'],
        14: ['color_yellow', 'emotion_happy'],
        15: ['color_yellow', 'emotion_sad'],
        16: ['color_yellow', 'emotion_happy'],
        17: ['color_yellow', 'emotion_happy'],
        18: ['color_yellow', 'emotion_happy'],
        19: ['color_yellow', 'emotion_happy'],
        20: ['color_yellow', 'emotion_happy'],
        21: ['color_yellow', 'emotion_happy'],
        22: ['color_yellow', 'emotion_surprise'],
        23: ['color_yellow', 'emotion_happy'],
        24: ['color_yellow', 'emotion_happy', 'object_heart'],
        25: ['color_yellow', 'emotion_neutral', 'object_heart'],
        26: ['color_yellow', 'emotion_happy', 'object_heart'],
        27: ['color_yellow', 'object_heart', 'mouth_kiss'],
        28: ['color_yellow', 'object_heart', 'mouth_kiss'],
        29: ['color_yellow', 'emotion_happy', 'object_heart', 'upside_down'],
        30: ['color_yellow', 'emotion_surprise', 'object_heart'],
        31: ['color_yellow', 'object_heart', 'mouth_kiss'],
        32: ['color_yellow', 'emotion_happy', 'emotion_surprise'],
        33: ['color_yellow', 'emotion_happy'],
        34: ['color_yellow', 'emotion_sad'],
        35: ['color_yellow', 'emotion_sad'],
        36: ['color_yellow', 'emotion_surprise'],
        37: ['color_yellow', 'emotion_surprise', 'emotion_neutral', 'upside_down'],
        38: ['color_yellow', 'emotion_neutral', 'emotion_shy'],
        39: ['color_yellow', 'emotion_surprise', 'emotion_shy'],
        40: ['color_yellow', 'object_sunglasses', 'emotion_happy'],
        41: ['color_yellow', 'object_domino_mask', 'emotion_evil'],
        42: ['color_yellow', 'object_domino_mask', 'emotion_evil', 'emotion_angry'],
        43: ['color_yellow', 'object_sick_mask'],
        44: ['color_yellow', 'object_halo', 'emotion_happy'],
        45: ['color_yellow', 'entity_clown', 'emotion_happy'],
        46: ['color_yellow', 'emotion_neutral', 'mouth_zigzag'],
        47: ['color_yellow', 'emotion_sad', 'effect_vomit'],
        48: ['color_yellow', 'emotion_happy', 'mouth_tongue'],
        49: ['color_yellow', 'eyes_xx', 'emotion_surprise'],
        50: ['color_yellow', 'emotion_angry'],
        51: ['color_yellow', 'emotion_happy'],
        52: ['color_yellow', 'emotion_laughing'],
        53: ['color_yellow', 'emotion_sad', 'emotion_shy'],
        54: ['color_yellow', 'emotion_evil', 'eyes_evil'],
        55: ['color_yellow', 'mouth_zigzag', 'eyes_xx'],
        56: ['color_yellow', 'emotion_surprise', 'effect_sweat'],
        57: ['color_yellow', 'emotion_neutral', 'object_glasses'],
        58: ['color_yellow', 'emotion_sad', 'eyes_closed'],
        59: ['color_yellow', 'eyes_roll', 'emotion_neutral'],
        60: ['color_yellow', 'emotion_happy', 'eyes_money'],
        61: ['color_yellow', 'emotion_laughing', 'eyes_closed'],
        62: ['color_yellow', 'emotion_sleepy', 'object_zz'],
        63: ['color_yellow', 'emotion_neutral', 'entity_no_face'],
        64: ['color_yellow', 'emotion_angry', 'effect_steam', 'eyes_closed'],
        65: ['color_yellow', 'emotion_surprise', 'effect_steam'],
        66: ['color_yellow', 'emotion_happy', 'object_glasses'],
        67: ['color_yellow', 'emotion_surprise', 'effect_sweat'],
        68: ['color_yellow', 'emotion_surprise', 'eyes_empty'],
        69: ['color_yellow', 'emotion_surprise', 'eyes_shine'],
        70: ['color_yellow', 'emotion_happy', 'eyes_shine'],
        71: ['color_yellow', 'emotion_laughing', 'eyes_shine'],
        72: ['color_yellow', 'emotion_angry', 'eyes_evil'],
        73: ['color_yellow', 'emotion_angry', 'eyes_evil'],
        74: ['color_yellow', 'emotion_evil', 'eyes_evil'],
        75: ['color_yellow', 'emotion_sad', 'mouth_bend'],
        76: ['color_yellow', 'emotion_surprise', 'eyebrow_raise'],
        77: ['color_yellow', 'emotion_neutral', 'eyes_wink', 'mouth_teeth'],
        78: ['color_yellow', 'emotion_neutral', 'entity_no_face'],
        79: ['color_yellow', 'emotion_angry', 'object_grawlix'],
        80: ['color_yellow', 'emotion_angry', 'emotion_sad', 'eyes_closed'],
        81: ['color_yellow', 'emotion_happy', 'entity_cyclops'],
        82: ['color_yellow', 'emotion_sad', 'entity_cyclops'],
        83: ['color_yellow', 'emotion_sad', 'object_bandage'],
        84: ['color_yellow', 'emotion_neutral', 'object_monocle'],
        85: ['color_yellow', 'emotion_happy', 'mouth_teeth', 'object_heart'],
        86: ['color_yellow', 'emotion_laughing', 'mouth_tongue'],
        87: ['color_yellow', 'emotion_sad', 'eyes_empty', 'mouth_zigzag'],
        88: ['color_yellow', 'emotion_sad', 'long_nose'],
        89: ['color_yellow', 'emotion_happy', 'long_nose'],
        90: ['color_yellow', 'emotion_evil', 'wide_nose'],
        91: ['color_yellow', 'emotion_sad'],
        92: ['color_yellow', 'entity_warning'],
        93: ['color_white', 'color_green', 'entity_tick'],
        94: ['color_white', 'color_red', 'entity_crossed_out'],
        95: ['color_white', 'color_red', 'object_heart'],
        96: ['color_red', 'emotion_cry', 'effect_tears'],
        97: ['color_red', 'emotion_neutral', 'effect_sweat'],
        98: ['color_red', 'emotion_evil'],
        99: ['color_red', 'emotion_angry', 'emotion_shy'],
        100: ['color_blue', 'emotion_happy', 'effect_freeze'],
        101: ['color_blue', 'emotion_neutral', 'effect_freeze', 'mouth_teeth'],
        102: ['color_blue', 'emotion_scared', 'effect_freeze', 'mouth_tongue'],
        103: ['color_blue', 'emotion_angry', 'eyes_closed', 'effect_freeze', 'mouth_tongue'],
        104: ['color_purple', 'emotion_angry', 'entity_devil', 'object_horns'],
        105: ['color_purple', 'emotion_happy', 'entity_devil', 'object_horns'],
        106: ['color_purple', 'emotion_evil', 'entity_devil', 'object_horns', 'eyes_evil'],
        107: ['color_purple', 'emotion_evil', 'emotion_happy', 'entity_devil', 'object_horns'],
        108: ['color_green', 'emotion_sick', 'mouth_zigzag'],
        109: ['color_green', 'emotion_sick'],
        110: ['color_green', 'emotion_sad', 'effect_vomit'],
        111: ['color_green', 'emotion_surprise', 'emotion_sad'],
        112: ['color_white', 'emotion_evil', 'eyes_evil'],
        113: ['color_white', 'emotion_sad'],
        114: ['color_white', 'emotion_sad', 'eyes_empty', 'wide_nose'],
        115: ['color_blue', 'entity_no_face', 'effect_freeze'],
        116: ['color_white', 'entity_no_face'],
        117: ['color_purple', 'entity_no_face', 'object_horns'],
        118: ['color_green', 'entity_no_face'],
        119: ['color_red', 'entity_no_face'],
        120: ['color_red', 'object_heart'],
        121: ['color_pink', 'object_heart'],
        122: ['color_blue', 'object_heart'],
        123: ['color_green', 'object_heart'],
        124: ['color_orange', 'object_heart'],
        125: ['color_purple', 'object_heart'],
        126: ['color_black', 'object_heart'],
        127: ['color_rainbow', 'object_heart'],
        128: ['color_yellow', 'emotion_happy'],
        129: ['color_yellow', 'emotion_laughing'],
        130: ['color_yellow', 'emotion_cry', 'effect_tears'],
        131: ['color_yellow', 'emotion_cry', 'effect_tears'],
        132: ['color_yellow', 'emotion_sad', 'eyes_closed', 'emotion_shy'],
        133: ['color_yellow', 'emotion_neutral'],
        134: ['color_yellow', 'emotion_neutral', 'eyes_closed'],
        135: ['color_yellow', 'emotion_neutral', 'upside_down'],
        136: ['color_yellow', 'emotion_happy', 'object_heart'],
        137: ['color_yellow', 'emotion_laughing', 'object_heart'],
        138: ['color_yellow', 'mouth_kiss', 'object_heart'],
        139: ['color_yellow', 'mouth_kiss', 'object_heart'],
        140: ['color_yellow', 'emotion_happy', 'eyes_closed'],
        141: ['color_yellow', 'mouth_kiss', 'emotion_neutral'],
        142: ['color_yellow', 'mouth_kiss', 'eyes_closed'],
        143: ['color_yellow', 'mouth_kiss', 'eyes_closed', 'object_heart'],
        144: ['color_yellow', 'emotion_happy', 'object_heart', 'eyes_closed'],
        145: ['color_yellow', 'emotion_happy', 'mouth_tongue'],
        146: ['color_yellow', 'emotion_goofy', 'emotion_happy', 'mouth_tongue'],
        147: ['color_yellow', 'emotion_happy', 'eyes_closed', 'mouth_tongue'],
        148: ['color_yellow', 'emotion_happy', 'eyes_closed', 'mouth_tongue'],
        149: ['color_yellow', 'emotion_happy', 'eyes_wink', 'mouth_tongue'],
        150: ['color_yellow', 'emotion_neutral', 'mouth_tongue'],
        151: ['color_yellow', 'emotion_happy', 'eyes_wink'],
        152: ['color_yellow', 'emotion_happy', 'emotion_goofy', 'mouth_tongue'],
        153: ['color_yellow', 'emotion_neutral', 'mouth_tongue'],
        154: ['color_yellow', 'emotion_surprise'],
        155: ['color_yellow', 'emotion_surprise', 'eyes_xx'],
        156: ['color_yellow', 'emotion_surprise'],
        157: ['color_yellow', 'emotion_surprise', 'eyes_closed'],
        158: ['color_yellow', 'emotion_surprise', 'eyes_closed'],
        159: ['color_yellow', 'emotion_sad'],
        160: ['color_yellow', 'emotion_surprise', 'mouth_tongue'],
        161: ['color_yellow', 'emotion_surprise', 'mouth_tongue', 'eyes_wide'],
        162: ['color_yellow', 'emotion_happy', 'upside_down'],
        163: ['color_yellow', 'emotion_happy', 'object_sunglasses'],
        164: ['color_yellow', 'emotion_sad', 'eyes_closed'],
        165: ['color_yellow', 'emotion_laughing', 'eyes_xx'],
        166: ['color_yellow', 'emotion_happy', 'emotion_shy'],
        167: ['color_yellow', 'emotion_happy', 'emotion_shy', 'eyes_closed'],
        168: ['color_yellow', 'emotion_surprise', 'emotion_neutral', 'emotion_shy', 'eyes_wide'],
        169: ['color_yellow', 'eyes_closed', 'mouth_kiss', 'emotion_shy'],
        170: ['color_yellow', 'mouth_drool'],
        171: ['color_yellow', 'eyes_roll', 'emotion_neutral'],
        172: ['color_yellow', 'mouth_zip'],
        173: ['color_yellow', 'eyes_closed', 'emotion_happy'],
        174: ['color_yellow', 'emotion_surprise'],
        175: ['color_yellow', 'emotion_surprise', 'eyes_wide'],
        176: ['color_yellow', 'object_sick_mask'],
        177: ['color_yellow', 'emotion_angry'],
        178: ['color_yellow', 'emotion_happy', 'eyes_closed', 'object_halo'],
        179: ['color_yellow', 'emotion_neutral', 'eyebrow_raise'],
        180: ['color_red', 'emotion_sad'],
        181: ['color_red', 'eyes_closed', 'emotion_sad'],
        182: ['color_red', 'emotion_angry', 'emotion_sad'],
        183: ['color_blue', 'emotion_surprise', 'effect_freeze'],
        184: ['color_yellow', 'emotion_laughing', 'effect_tears'],
        185: ['color_purple', 'emotion_evil', 'object_horns', 'eyes_closed'],
        186: ['color_purple', 'emotion_sad', 'object_horns', 'eyes_closed'],
        187: ['color_green', 'emotion_sad', 'mouth_zigzag'],
        188: ['color_green', 'emotion_sad'],
        189: ['color_yellow', 'emotion_laughing', 'eyes_closed'],
        190: ['color_yellow', 'emotion_sad'],
        191: ['color_yellow', 'emotion_smirk'],
        192: ['color_yellow', 'emotion_sad'],
        193: ['color_yellow', 'emotion_angry'],
        194: ['color_yellow', 'emotion_sick', 'object_thermometer', 'emotion_shy'],
        195: ['color_yellow', 'emotion_sad', 'object_bandage'],
        196: ['color_yellow', 'emotion_surprise', 'eyes_xx'],
        197: ['color_yellow', 'emotion_happy', 'eyes_money', 'mouth_tongue'],
        198: ['color_purple', 'emotion_happy', 'entity_devil'],
        199: ['color_purple', 'emotion_sad', 'entity_devil'],
        200: ['color_purple', 'emotion_happy', 'entity_poop'],
        201: ['color_white', 'entity_skull'],
        202: ['color_red', 'entity_robot', 'emotion_neutral'],
        203: ['color_green', 'emotion_happy', 'entity_alien'],
        204: ['color_red', 'emotion_sad', 'emotion_angry', 'eyes_closed'],
        205: ['color_orange', 'entity_pumpkin'],
        206: ['color_yellow', 'emotion_sad'],
        207: ['color_yellow', 'entity_no_face'],
        208: ['color_yellow', 'emotion_surprise'],
        209: ['color_green', 'emotion_sick'],
        210: ['color_green', 'emotion_sad', 'effect_vomit'],
        211: ['color_yellow', 'emotion_sad'],
        212: ['color_blue', 'emotion_sad'],
        213: ['color_yellow', 'emotion_neutral', 'object_zz'],
        214: ['color_yellow', 'emotion_surprise', 'emotion_neutral'],
        215: ['color_purple', 'emotion_neutral', 'object_zz'],
        216: ['color_yellow', 'emotion_neutral', 'object_zz', 'eyes_closed'],
        217: ['color_yellow', 'emotion_neutral', 'mouth_drool', 'eyes_closed'],
        218: ['color_blue', 'emotion_neutral', 'mouth_drool', 'eyes_closed'],
        219: ['color_yellow', 'emotion_happy', 'emotion_shy'],
        220: ['color_yellow', 'emotion_happy', 'object_glasses', 'object_hand'],
        221: ['color_yellow', 'emotion_happy', 'eyebrow_raise'],
        222: ['color_yellow', 'emotion_surprise', 'object_monocle'],
        223: ['color_yellow', 'emotion_sad', 'eyes_empty'],
        224: ['color_yellow', 'emotion_sad', 'effect_vomit'],
        225: ['color_yellow', 'emotion_sad'],
        226: ['color_yellow', 'emotion_smirk', 'eyes_roll'],
        227: ['color_yellow', 'emotion_neutral', 'eyes_empty', 'mouth_zip'],
        228: ['color_yellow', 'emotion_happy', 'entity_clown'],
        229: ['color_yellow', 'emotion_surprise', 'entity_clown'],
        230: ['color_white', 'emotion_happy', 'entity_clown'],
        231: ['color_yellow', 'emotion_surprise'],
        232: ['color_yellow', 'emotion_neutral', 'effect_sweat'],
        233: ['color_yellow', 'emotion_happy', 'eyes_star'],
        234: ['color_yellow', 'emotion_happy', 'object_hat_party'],
        235: ['color_yellow', 'emotion_surprise', 'object_halo'],
        236: ['color_yellow', 'emotion_sad', 'effect_vomit'],
        237: ['color_yellow', 'emotion_evil', 'object_horns'],
        238: ['color_yellow', 'emotion_happy', 'object_hat_cowboy'],
        239: ['color_yellow', 'emotion_happy', 'object_heart'],
        240: ['color_yellow', 'mouth_kiss'],
        241: ['color_yellow', 'eyes_closed', 'mouth_kiss', 'emotion_shy'],
        242: ['color_yellow', 'emotion_scared', 'object_hand'],
        243: ['color_yellow', 'emotion_sad', 'eyes_closed'],
        244: ['color_yellow', 'emotion_surprise', 'object_hand'],
        245: ['color_yellow', 'emotion_sad', 'eyes_closed'],
        246: ['color_yellow', 'eyes_closed', 'emotion_happy', 'emotion_shy'],
        247: ['color_yellow', 'emotion_surprise'],
        248: ['color_yellow', 'emotion_happy', 'eyes_closed', 'emotion_shy', 'object_hand'],
        249: ['color_yellow', 'emotion_sad', 'effect_tears'],
        250: ['color_yellow', 'emotion_sad', 'effect_tears'],
        251: ['color_yellow', 'eyes_closed', 'emotion_laughing'],
        252: ['color_yellow', 'eyes_closed', 'emotion_laughing'],
        253: ['color_yellow', 'emotion_laughing', 'effect_tears'],
        254: ['color_yellow', 'eyes_closed', 'effect_sweat', 'emotion_laughing'],
        255: ['color_yellow', 'emotion_happy', 'effect_tears'],
        256: ['color_yellow', 'eyes_closed', 'emotion_happy'],
        257: ['color_yellow', 'eyes_closed', 'emotion_happy', 'object_heart'],
        258: ['color_yellow', 'emotion_happy'],
        259: ['color_yellow', 'eyes_closed', 'emotion_happy'],
        260: ['color_yellow', 'eyes_closed', 'emotion_happy', 'mouth_tongue'],
        261: ['color_yellow', 'emotion_goofy', 'emotion_laughing', 'mouth_tongue'],
        262: ['color_yellow', 'emotion_smirk', 'object_sunglasses'],
        263: ['color_yellow', 'emotion_neutral', 'object_glasses'],
        264: ['color_yellow', 'emotion_goofy', 'object_glasses'],
        265: ['color_yellow', 'eyes_star', 'emotion_laughing'],
        266: ['color_yellow', 'object_hat_party', 'eyes_closed', 'emotion_happy'],
        267: ['color_yellow', 'emotion_sad', 'eyes_roll'],
        268: ['color_yellow', 'emotion_sad', 'eyes_closed'],
        269: ['color_yellow', 'emotion_happy', 'emotion_evil'],
        270: ['color_yellow', 'eyes_closed', 'emotion_sad'],
        271: ['color_yellow', 'eyes_closed', 'emotion_sad'],
        272: ['color_yellow', 'emotion_neutral', 'mouth_bend'],
        273: ['color_yellow', 'emotion_angry', 'mouth_zigzag'],
        274: ['color_yellow', 'emotion_happy'],
        275: ['color_yellow', 'emotion_sad', 'mouth_bend'],
        276: ['color_yellow', 'emotion_sad', 'effect_tears'],
        277: ['color_yellow', 'emotion_cry', 'effect_tears'],
        278: ['color_yellow', 'eyes_closed', 'emotion_neutral'],
        279: ['color_yellow', 'emotion_sad', 'emotion_angry'],
        280: ['color_yellow', 'emotion_happy'],
        281: ['color_yellow', 'emotion_angry', 'effect_steam'],
        282: ['color_yellow', 'emotion_surprise', 'effect_exploding_head'],
        283: ['color_red', 'emotion_sad', 'effect_sweat'],
        284: ['color_blue', 'emotion_sad', 'effect_freeze'],
        285: ['color_yellow', 'emotion_surprise', 'emotion_shy'],
        286: ['color_white', 'effect_steam'],
        287: ['color_yellow', 'emotion_surprise', 'effect_sweat', 'object_hand'],
        288: ['color_yellow', 'entity_no_face', 'style_outline'],
        289: ['color_yellow', 'emotion_sad', 'entity_cyclops'],
        290: ['color_yellow', 'long_nose', 'emotion_neutral'],
        291: ['color_yellow', 'emotion_happy', 'emotion_laughing'],
        292: ['color_yellow', 'emotion_neutral', 'object_hand'],
        293: ['color_yellow', 'emotion_goofy', 'mouth_bend', 'emotion_neutral'],
        294: ['color_yellow', 'emotion_goofy', 'mouth_bend', 'object_hand'],
        295: ['color_yellow', 'emotion_neutral'],
        296: ['color_yellow', 'emotion_neutral', 'eyes_roll'],
        297: ['color_yellow', 'emotion_yawn'],
        298: ['color_yellow', 'emotion_yawn', 'object_hand'],
        299: ['color_yellow', 'eyes_closed', 'emotion_sleepy'],
        300: ['color_yellow', 'emotion_dizzy', 'eyes_spiral'],
        301: ['color_yellow', 'emotion_shy', 'eyes_closed', 'effect_runny_nose'],
    }
    num_sprites_original = len(sprites)
    labels_original = np.zeros((num_sprites_original, N_CLASSES), dtype=np.float32)

    print("Processing labels from the in-code dictionary...")
    for image_index, label_list in image_label_map.items():
        if image_index < num_sprites_original:
            for label_name in label_list:
                try:
                    label_index = LABEL_NAMES.index(label_name)
                    labels_original[image_index, label_index] = 1.0
                except ValueError:
                    print(f"Warning: The label '{label_name}' for image {image_index} is not in LABEL_NAMES. It will be ignored.")

    # ==========================================================================
    # NEW: DATA AUGMENTATION SECTION
    # ==========================================================================
    print("\n--- Starting Data Augmentation to Balance Colors ---")

    augmented_sprites = list(sprites)
    augmented_labels = list(labels_original)

    # Define how many times to duplicate samples of a given color
    # We want to bring their counts closer to the number of yellow emojis.
    augmentation_map = {
        'color_red': 10,
        'color_green': 10,
        'color_blue': 12,
        'color_purple': 10,
        'color_white': 8,
        'color_pink': 15,
        'color_orange': 15,
        'color_brown': 15,
        'color_black': 15,
    }

    for color, multiplier in augmentation_map.items():
        if color in LABEL_NAMES:
            color_index = LABEL_NAMES.index(color)
            # Find all images that have this color label
            indices_to_duplicate = np.where(labels_original[:, color_index] == 1.0)[0]

            if len(indices_to_duplicate) > 0:
                print(f"Duplicating {len(indices_to_duplicate)} '{color}' images {multiplier-1} times.")
                for _ in range(multiplier - 1): # Duplicate N-1 times to get N total copies
                    for i in indices_to_duplicate:
                        augmented_sprites.append(sprites[i])
                        augmented_labels.append(labels_original[i])

    final_sprites = np.array(augmented_sprites)
    final_labels = np.array(augmented_labels)

    print(f"\nOriginal dataset size: {len(sprites)}")
    print(f"Augmented dataset size: {len(final_sprites)}")
    # ==========================================================================

    # --- Save the final augmented arrays ---
    np.save('emojis.npy', final_sprites)
    np.save('labels.npy', final_labels)
    print("Augmented dataset processed and saved to emojis.npy and labels.npy.")

    return final_sprites, final_labels


class CustomEmojiDataset(Dataset):
    def __init__(self, sprites_path='emojis.npy', labels_path='labels.npy', transform=None):
        self.sprites = np.load(sprites_path)
        self.labels = np.load(labels_path)
        self.transform = transform
        print(f"Dataset loaded. Sprites shape: {self.sprites.shape}, Labels shape: {self.labels.shape}")

    def __len__(self):
        return len(self.sprites)

    def __getitem__(self, idx):
        # Convert sprite from (H, W, C) numpy array to (C, H, W) tensor
        sprite = Image.fromarray(self.sprites[idx])
        if self.transform:
            image = self.transform(sprite)
        else:
            image = transforms.ToTensor()(sprite)

        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

In [ ]:
# ==============================================================================
# 3. MODEL ARCHITECTURE
# This is the UNet-based model code you provided.
# ==============================================================================
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, is_res: bool = False):
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.is_res:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            if self.same_channels:
                out = x + x2
            else:
                shortcut = nn.Conv2d(x.shape[1], x2.shape[1], kernel_size=1, stride=1, padding=0).to(x.device)
                out = shortcut(x) + x2
            return out / 1.414
        else:
            x1 = self.conv1(x)
            x2 = self.conv2(x1)
            return x2

class UnetUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetUp, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        )

    def forward(self, x, skip):
        x = torch.cat((x, skip), 1)
        x = self.model(x)
        return x

class UnetDown(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UnetDown, self).__init__()
        self.model = nn.Sequential(
            ResidualConvBlock(in_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.model(x)

class EmbedFC(nn.Module):
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        self.input_dim = input_dim
        self.model = nn.Sequential(
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

class ContextUnet(nn.Module):
    def __init__(self, in_channels, n_feat=256, n_cfeat=10, height=28):
        super(ContextUnet, self).__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_cfeat = n_cfeat
        self.h = height

        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)
        self.to_vec = nn.Sequential(nn.AvgPool2d(4), nn.GELU()) # 16 -> 8 -> 4, so pool with 4

        self.timeembed1 = EmbedFC(1, 2 * n_feat)
        self.timeembed2 = EmbedFC(1, 1 * n_feat)
        self.contextembed1 = EmbedFC(n_cfeat, 2 * n_feat)
        self.contextembed2 = EmbedFC(n_cfeat, 1 * n_feat)

        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, self.h // 4, self.h // 4),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, t, c=None):
        x = self.init_conv(x)
        down1 = self.down1(x)
        down2 = self.down2(down1)
        hiddenvec = self.to_vec(down2)

        if c is None:
            c = torch.zeros(x.shape[0], self.n_cfeat).to(x)

        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        up1 = self.up0(hiddenvec)
        up2 = self.up1(cemb1 * up1 + temb1, down2)
        up3 = self.up2(cemb2 * up2 + temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out

In [ ]:
# ==============================================================================
# 4. TRAINING SETUP & EXECUTION
# ==============================================================================

# --- Hyperparameters ---
timesteps = 500
beta1 = 1e-4
beta2 = 0.02
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
n_feat = 256  # Increased features for more capacity
height = 16
save_dir = '/content/weights/'
batch_size = 32
n_epoch = 200
lrate = 3e-4

# --- Diffusion Components ---
b_t = (beta2 - beta1) * torch.linspace(0, 1, timesteps + 1, device=device) + beta1
a_t = 1 - b_t
ab_t = torch.cumsum(a_t.log(), dim=0).exp()
ab_t[0] = 1

def perturb_input(x, t, noise):
    """Adds noise to the input image x at timestep t."""
    return ab_t.sqrt()[t, None, None, None] * x + (1 - ab_t[t, None, None, None]).sqrt() * noise

# --- Main Training Function ---
def train_model():
    print(f"Using device: {device}")
    os.makedirs(save_dir, exist_ok=True)

    # Dataset and Dataloader
    transform = transforms.Compose([
        transforms.ToTensor(),                # from [0,255] to range [0.0,1.0]
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # range [-1,1]
    ])
    dataset = CustomEmojiDataset(transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    # Model and Optimizer
    nn_model = ContextUnet(in_channels=3, n_feat=n_feat, n_cfeat=N_CLASSES, height=height).to(device)
    optim = torch.optim.Adam(nn_model.parameters(), lr=lrate)

    # Training loop
    nn_model.train()
    for ep in range(n_epoch):
        print(f'Epoch {ep+1}/{n_epoch}')
        optim.param_groups[0]['lr'] = lrate * (1 - ep / n_epoch)
        pbar = tqdm(dataloader, mininterval=2)
        total_loss = 0

        for x, c in pbar:
            optim.zero_grad()
            x = x.to(device)
            c = c.to(device)

            # Use context dropout for classifier-free guidance
            context_mask = torch.bernoulli(torch.ones(c.shape[0]) * 0.9).to(device)
            c = c * context_mask.unsqueeze(-1)

            noise = torch.randn_like(x)
            t = torch.randint(1, timesteps + 1, (x.shape[0],)).to(device)
            x_pert = perturb_input(x, t, noise)

            pred_noise = nn_model(x_pert, t / timesteps, c=c)
            loss = F.mse_loss(pred_noise, noise)
            loss.backward()
            optim.step()

            total_loss += loss.item()
            pbar.set_description(f"loss: {loss.item():.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"Average loss for epoch {ep+1}: {avg_loss:.4f}")

        if (ep + 1) % 10 == 0 or ep == n_epoch - 1:
            torch.save(nn_model.state_dict(), os.path.join(save_dir, f"context_model_{ep+1}.pth"))
            print(f"Saved model checkpoint to {save_dir}context_model_{ep+1}.pth")
    print("Training complete.")

In [ ]:
# ==============================================================================
# 5. INFERENCE & GENERATION
# ==============================================================================

def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t, None, None, None] * z
    term1 = (1 - a_t[t, None, None, None]) / (1 - ab_t[t, None, None, None]).sqrt()
    mean = (x - pred_noise * term1) / a_t[t, None, None, None].sqrt()
    return mean + noise

@torch.no_grad()
def sample_ddpm_context(model, n_sample, context, w=2.0):
    """
    Sample from the model using classifier-free guidance.
    w: guidance weight. w=0 is unconditional, w>1 strengthens condition.
    """
    samples = torch.randn(n_sample, 3, height, height).to(device)
    c_uncond = torch.zeros(n_sample, N_CLASSES).to(device) # Unconditional context

    for i in range(timesteps, 0, -1):
        # print(f'sampling timestep {i:3d}', end='\r')
        t = torch.full((n_sample,), i / timesteps, device=device)
        t_int = torch.full((n_sample,), i, device=device, dtype=torch.long)

        # Predict noise for both conditional and unconditional
        pred_noise_cond = model(samples, t, c=context)
        pred_noise_uncond = model(samples, t, c=c_uncond)

        # Classifier-free guidance formula
        eps = (1 + w) * pred_noise_cond - w * pred_noise_uncond

        z = torch.randn_like(samples) if i > 1 else 0
        samples = denoise_add_noise(samples, t_int, eps, z)

    return samples

def show_images(imgs, contexts, title="Generated Emojis"):
    """
    Helper to plot generated images in a grid, with each image labeled
    by its generation context.
    """
    # De-normalize images from [-1, 1] to [0, 1]
    imgs = (imgs.clamp(-1, 1) + 1) / 2

    n_cols = 8
    n_rows = (imgs.shape[0] + n_cols - 1) // n_cols

    # Create a figure and a set of subplots
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 2.5))

    # Flatten the axes array for easy iteration
    axs = axs.flatten()

    for i in range(len(imgs)):
        ax = axs[i]

        # Display the image
        ax.imshow(imgs[i].detach().cpu().permute(1, 2, 0))
        ax.axis('off') # Hide axes ticks

        # --- Generate the label string from the context vector ---
        context_vector = contexts[i]
        # Find the indices of active labels (where value is > 0.5)
        active_indices = torch.where(context_vector > 0.5)[0]
        # Map indices to label names
        label_list = [LABEL_NAMES[j] for j in active_indices]
        # Join them into a title string
        label_title = "\n".join(label_list).replace("_", " ").title()

        ax.set_title(label_title, fontsize=8)

    # Hide any unused subplots
    for i in range(len(imgs), len(axs)):
        axs[i].axis('off')

    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle
    plt.show()


def generate_emojis(model_path):
    """
    Loads a trained model and generates emojis based on custom contexts,
    then displays them with their corresponding labels.
    """
    print("--- Starting Generation ---")
    nn_model = ContextUnet(in_channels=3, n_feat=n_feat, n_cfeat=N_CLASSES, height=height).to(device)
    nn_model.load_state_dict(torch.load(model_path, map_location=device))
    nn_model.eval()
    print(f"Loaded model weights from {model_path}")

    n_samples_per_prompt = 8
    all_contexts = []

    # --- Define your generation prompts here using the NEW, EXPANDED labels ---

    # Prompt 1: Yellow emoji, laughing with tears effect
    ctx1 = torch.zeros(1, N_CLASSES)
    ctx1[0, LABEL_NAMES.index('color_yellow')] = 1.0
    ctx1[0, LABEL_NAMES.index('emotion_laughing')] = 1.0
    ctx1[0, LABEL_NAMES.index('effect_tears')] = 1.0  # Corrected from 'tears'
    all_contexts.append(ctx1.repeat(n_samples_per_prompt, 1))

    # Prompt 2: Purple devil, evil and angry with horns
    ctx2 = torch.zeros(1, N_CLASSES)
    ctx2[0, LABEL_NAMES.index('color_purple')] = 1.0
    ctx2[0, LABEL_NAMES.index('entity_devil')] = 1.0
    ctx2[0, LABEL_NAMES.index('emotion_angry')] = 1.0
    ctx2[0, LABEL_NAMES.index('object_horns')] = 1.0
    all_contexts.append(ctx2.repeat(n_samples_per_prompt, 1))

    # Prompt 3: Freezing blue face, scared
    ctx3 = torch.zeros(1, N_CLASSES)
    ctx3[0, LABEL_NAMES.index('color_blue')] = 1.0
    ctx3[0, LABEL_NAMES.index('emotion_scared')] = 1.0
    ctx3[0, LABEL_NAMES.index('effect_freeze')] = 1.0 # Corrected from 'freeze'
    all_contexts.append(ctx3.repeat(n_samples_per_prompt, 1))

    # Prompt 4: Dizzy yellow emoji with spiral eyes
    ctx4 = torch.zeros(1, N_CLASSES)
    ctx4[0, LABEL_NAMES.index('color_yellow')] = 1.0
    ctx4[0, LABEL_NAMES.index('emotion_dizzy')] = 1.0
    ctx4[0, LABEL_NAMES.index('eyes_spiral')] = 1.0
    all_contexts.append(ctx4.repeat(n_samples_per_prompt, 1))

    # Prompt 5: Surprised emoji with an exploding head
    ctx5 = torch.zeros(1, N_CLASSES)
    ctx5[0, LABEL_NAMES.index('color_yellow')] = 1.0
    ctx5[0, LABEL_NAMES.index('emotion_surprise')] = 1.0
    ctx5[0, LABEL_NAMES.index('effect_exploding_head')] = 1.0
    all_contexts.append(ctx5.repeat(n_samples_per_prompt, 1))

    # Prompt 6: A neutral red robot face
    ctx6 = torch.zeros(1, N_CLASSES)
    ctx6[0, LABEL_NAMES.index('color_red')] = 1.0
    ctx6[0, LABEL_NAMES.index('entity_robot')] = 1.0
    ctx6[0, LABEL_NAMES.index('emotion_neutral')] = 1.0
    all_contexts.append(ctx6.repeat(n_samples_per_prompt, 1))

    # Combine all contexts
    final_contexts = torch.cat(all_contexts).to(device)

    print("Generating images with custom contexts...")
    # Increase guidance weight 'w' for stronger adherence to prompts.
    # You can play with this value; higher values make it follow the prompt more strictly.
    samples = sample_ddpm_context(nn_model, final_contexts.shape[0], final_contexts, w=5.0)

    # Call the show_images function, passing the contexts as well
    show_images(samples, final_contexts.cpu(), title="Generated Emojis with New Prompts")

In [ ]:
# Step 1: Prepare the dataset
prepare_dataset()

Fetching dataset info from GitHub API: https://api.github.com/repos/cbarkinozer/DataScience/contents/LargeLanguageModels/Emoji1
Found 302 emoji images. Downloading...


Processing labels from the in-code dictionary...

--- Starting Data Augmentation to Balance Colors ---
Duplicating 14 'color_red' images 9 times.
Duplicating 12 'color_green' images 9 times.
Duplicating 10 'color_blue' images 11 times.
Duplicating 12 'color_purple' images 9 times.
Duplicating 10 'color_white' images 7 times.
Duplicating 1 'color_pink' images 14 times.
Duplicating 2 'color_orange' images 14 times.
Duplicating 1 'color_black' images 14 times.

Original dataset size: 302
Augmented dataset size: 880
Augmented dataset processed and saved to emojis.npy and labels.npy.


(array([[[[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
          ...,
          [  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0]],
 
         [[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
          ...,
          [  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0]],
 
         [[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
          ...,
          [  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0]],
 
         ...,
 
         [[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
          ...,
          [  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0]],
 
         [[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
          ...,
          [  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0]],
 
         [[  0,   0,   0],
          [  0,   0,   0],
          [  0,   0,   0],
   

In [ ]:
# Step 2: Train the model
train_model()

Using device: cuda:0
Dataset loaded. Sprites shape: (880, 16, 16, 3), Labels shape: (880, 88)
Epoch 1/200


loss: 0.2534: 100%|██████████| 28/28 [00:02<00:00, 12.48it/s]


Average loss for epoch 1: 0.5864
Epoch 2/200


loss: 0.2634: 100%|██████████| 28/28 [00:02<00:00, 12.76it/s]


Average loss for epoch 2: 0.2854
Epoch 3/200


...

Average loss for epoch 198: 0.0370
Epoch 199/200


loss: 0.0247: 100%|██████████| 28/28 [00:02<00:00, 11.73it/s]


Average loss for epoch 199: 0.0402
Epoch 200/200


loss: 0.0442: 100%|██████████| 28/28 [00:02<00:00, 11.75it/s]


Average loss for epoch 200: 0.0350
Saved model checkpoint to /content/weights/context_model_200.pth
Training complete.


In [ ]:
# Step 3: Generate new emojis using the trained model
final_model_path = os.path.join(save_dir, f"context_model_{n_epoch}.pth")
if os.path.exists(final_model_path):
    generate_emojis(model_path=final_model_path)
else:
    print("Could not find trained model. Please ensure training completed successfully.")

--- Starting Generation ---
Loaded model weights from /content/weights/context_model_200.pth
Generating images with custom contexts...
